# Soccer dataset 

In [16]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

# =====================================================================
# Project paths
# =====================================================================
# This makes paths work whether the notebook kernel starts in the project
# root or inside the notebooks folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "database.sqlite").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
db_path = DATA_DIR / "database.sqlite" 

# Safety check to prevent SQLite from creating an empty database
if (not db_path.exists()) or db_path.stat().st_size == 0:
    raise FileNotFoundError(
        f"Database not found at: {db_path.resolve()}\n"
        "SQLite may create an empty file if the path is wrong."
    )

# Create the SQLite connection
conn = sqlite3.connect(str(db_path))
print(f"Successfully connected to {db_path} via sqlite3!")

# Query the master table for all user table names
tables_query = """
    SELECT name 
    FROM sqlite_master 
    WHERE type='table';
"""
tables_df = pd.read_sql(tables_query, conn)

print("Tables in the SQLite Database:")
display(tables_df)

countries_leagues_query = """
    SELECT Country.name as CountryName, 
           League.name as LeagueName 
    FROM League
    JOIN Country ON Country.id = League.country_id;
"""

leagues_df = pd.read_sql(countries_leagues_query, conn)
display(leagues_df)


Successfully connected to c:\Users\Madsl\Dropbox\DTU\Kand\3.Semester\MBML\Project_MBML\data\database.sqlite via sqlite3!
Tables in the SQLite Database:


,name
0,sqlite_sequence
1,Player_Attributes
2,Player
3,Match
4,League
5,Country
6,Team
7,Team_Attributes


,CountryName,LeagueName
0,Belgium,Belgium Jupiler League
1,England,England Premier League
2,France,France Ligue 1
3,Germany,Germany 1. Bundesliga
4,Italy,Italy Serie A
5,Netherlands,Netherlands Eredivisie
6,Poland,Poland Ekstraklasa
7,Portugal,Portugal Liga ZON Sagres
8,Scotland,Scotland Premier League
9,Spain,Spain LIGA BBVA


In [17]:
# Extract only the English Premier League matches
# We join the Match table with the League and Team tables to get readable names
epl_query = """
    SELECT 
        M.season, 
        M.stage,
        M.date,
        HT.team_long_name AS home_team,
        AT.team_long_name AS away_team,
        M.home_team_goal, 
        M.away_team_goal
    FROM Match M
    JOIN League L ON M.league_id = L.id
    JOIN Team HT ON HT.team_api_id = M.home_team_api_id
    JOIN Team AT ON AT.team_api_id = M.away_team_api_id
    WHERE L.name = 'England Premier League'
    ORDER BY M.date DESC;
"""

epl_matches = pd.read_sql(epl_query, conn)

print(f"Total Premier League matches found: {len(epl_matches)}")
display(epl_matches.head(10))

#save the dataframe to a csv file
epl_matches.to_csv(DATA_DIR / "epl_matches.csv", index=False)
print(f"Saved EPL matches to {DATA_DIR / 'epl_matches.csv'}")


Total Premier League matches found: 3040


,season,stage,date,home_team,away_team,home_team_goal,away_team_goal
0,2015/2016,38,2016-05-17 00:00:00,Manchester United,Bournemouth,3,1
1,2015/2016,38,2016-05-15 00:00:00,Arsenal,Aston Villa,4,0
2,2015/2016,38,2016-05-15 00:00:00,Chelsea,Leicester City,1,1
3,2015/2016,38,2016-05-15 00:00:00,Everton,Norwich City,3,0
4,2015/2016,38,2016-05-15 00:00:00,Newcastle United,Tottenham Hotspur,5,1
5,2015/2016,38,2016-05-15 00:00:00,Southampton,Crystal Palace,4,1
6,2015/2016,38,2016-05-15 00:00:00,Stoke City,West Ham United,2,1
7,2015/2016,38,2016-05-15 00:00:00,Swansea City,Manchester City,1,1
8,2015/2016,38,2016-05-15 00:00:00,Watford,Sunderland,2,2
9,2015/2016,38,2016-05-15 00:00:00,West Bromwich Albion,Liverpool,1,1


Saved EPL matches to c:\Users\Madsl\Dropbox\DTU\Kand\3.Semester\MBML\Project_MBML\data\epl_matches.csv


In [18]:
#load csv and display
import pandas as pd
epl_matches_loaded = pd.read_csv(DATA_DIR / "epl_matches.csv")
display(epl_matches_loaded.head(40))

# list seasons and number of matches per season
season_counts = epl_matches_loaded['season'].value_counts().sort_index()
print("Number of matches per season:")
display(season_counts)


,season,stage,date,home_team,away_team,home_team_goal,away_team_goal
0,2015/2016,38,2016-05-17 00:00:00,Manchester United,Bournemouth,3,1
1,2015/2016,38,2016-05-15 00:00:00,Arsenal,Aston Villa,4,0
2,2015/2016,38,2016-05-15 00:00:00,Chelsea,Leicester City,1,1
3,2015/2016,38,2016-05-15 00:00:00,Everton,Norwich City,3,0
4,2015/2016,38,2016-05-15 00:00:00,Newcastle United,Tottenham Hotspur,5,1
5,2015/2016,38,2016-05-15 00:00:00,Southampton,Crystal Palace,4,1
6,2015/2016,38,2016-05-15 00:00:00,Stoke City,West Ham United,2,1
7,2015/2016,38,2016-05-15 00:00:00,Swansea City,Manchester City,1,1
8,2015/2016,38,2016-05-15 00:00:00,Watford,Sunderland,2,2
9,2015/2016,38,2016-05-15 00:00:00,West Bromwich Albion,Liverpool,1,1


Number of matches per season:


season
2008/2009    380
2009/2010    380
2010/2011    380
2011/2012    380
2012/2013    380
2013/2014    380
2014/2015    380
2015/2016    380
Name: count, dtype: int64

## Player data in the SQLite database

The player information lives in three places:

- `Player`: one row per player with IDs, name, birthday, height, and weight.
- `Player_Attributes`: repeated FIFA-style ratings over time for each player, dated from 2007-02-22 to 2016-07-07.
- `Match`: line-up player IDs (`home_player_1` ... `away_player_11`) plus on-pitch X/Y coordinates for formations.

Quick database-wide facts from inspection: 11,060 players, 183,978 player attribute snapshots, and every player has at least one attribute row. In Premier League lineups specifically, there are 1,397 distinct players and all of them have attribute rows.


In [19]:
# Inspect the player-related table schemas
player_related_tables = ["Player", "Player_Attributes", "Match"]

schema_rows = []
for table in player_related_tables:
    table_schema = pd.read_sql(f"PRAGMA table_info({table});", conn)
    table_schema.insert(0, "table", table)
    schema_rows.append(table_schema)

player_schema = pd.concat(schema_rows, ignore_index=True)
display(player_schema)


,table,cid,name,type,notnull,dflt_value,pk
0,Player,0,id,INTEGER,0,None,1
1,Player,1,player_api_id,INTEGER,0,None,0
2,Player,2,player_name,TEXT,0,None,0
3,Player,3,player_fifa_api_id,INTEGER,0,None,0
4,Player,4,birthday,TEXT,0,None,0
...,...,...,...,...,...,...,...
159,Match,110,GBD,NUMERIC,0,None,0
160,Match,111,GBA,NUMERIC,0,None,0
161,Match,112,BSH,NUMERIC,0,None,0
162,Match,113,BSD,NUMERIC,0,None,0


In [20]:
# High-level counts and date ranges for the player data
player_overview = pd.read_sql(
    '''
    SELECT
        (SELECT COUNT(*) FROM Player) AS n_players,
        (SELECT COUNT(*) FROM Player_Attributes) AS n_player_attribute_rows,
        (SELECT COUNT(DISTINCT player_api_id) FROM Player_Attributes) AS n_players_with_attributes,
        (SELECT MIN(birthday) FROM Player) AS earliest_birthday,
        (SELECT MAX(birthday) FROM Player) AS latest_birthday,
        (SELECT MIN(date) FROM Player_Attributes) AS first_attribute_date,
        (SELECT MAX(date) FROM Player_Attributes) AS last_attribute_date
    ''',
    conn,
)
display(player_overview)

attribute_rows_per_player = pd.read_sql(
    '''
    SELECT
        COUNT(*) AS n_players,
        MIN(n_attribute_rows) AS min_attribute_rows,
        ROUND(AVG(n_attribute_rows), 2) AS avg_attribute_rows,
        MAX(n_attribute_rows) AS max_attribute_rows
    FROM (
        SELECT player_api_id, COUNT(*) AS n_attribute_rows
        FROM Player_Attributes
        GROUP BY player_api_id
    )
    ''',
    conn,
)
display(attribute_rows_per_player)


,n_players,n_player_attribute_rows,n_players_with_attributes,earliest_birthday,latest_birthday,first_attribute_date,last_attribute_date
0,11060,183978,11060,1967-01-23 00:00:00,1999-04-24 00:00:00,2007-02-22 00:00:00,2016-07-07 00:00:00


,n_players,min_attribute_rows,avg_attribute_rows,max_attribute_rows
0,11060,2,16.63,56


In [21]:
# What does a player row contain, and are there missing values?
players = pd.read_sql("SELECT * FROM Player;", conn, parse_dates=["birthday"])
display(players.head())

player_missingness = (
    players.isna()
    .sum()
    .rename("missing_values")
    .reset_index()
    .rename(columns={"index": "column"})
)
player_missingness["missing_pct"] = (
    100 * player_missingness["missing_values"] / len(players)
).round(2)
display(player_missingness)

display(players[["height", "weight"]].describe().round(2))


,id,player_api_id,player_name,player_fifa_api_id,birthday,height,weight
0,1,505942,Aaron Appindangoye,218353,1992-02-29,182.88,187
1,2,155782,Aaron Cresswell,189615,1989-12-15,170.18,146
2,3,162549,Aaron Doran,186170,1991-05-13,170.18,163
3,4,30572,Aaron Galindo,140161,1982-05-08,182.88,198
4,5,23780,Aaron Hughes,17725,1979-11-08,182.88,154


,column,missing_values,missing_pct
0,id,0,0.0
1,player_api_id,0,0.0
2,player_name,0,0.0
3,player_fifa_api_id,0,0.0
4,birthday,0,0.0
5,height,0,0.0
6,weight,0,0.0


,height,weight
count,11060.00,11060.00
mean,181.87,168.38
std,6.37,14.99
min,157.48,117.00
25%,177.80,159.00
50%,182.88,168.00
75%,185.42,179.00
max,208.28,243.00


In [22]:
# What does an attribute snapshot contain?
player_attributes = pd.read_sql(
    "SELECT * FROM Player_Attributes;",
    conn,
    parse_dates=["date"],
)

display(player_attributes.head())

rating_columns = [
    col for col in player_attributes.columns
    if col not in [
        "id",
        "player_fifa_api_id",
        "player_api_id",
        "date",
        "preferred_foot",
        "attacking_work_rate",
        "defensive_work_rate",
    ]
]

display(player_attributes[rating_columns].describe().T.round(2))
display(player_attributes["preferred_foot"].value_counts(dropna=False).rename_axis("preferred_foot").reset_index(name="rows"))
display(player_attributes["attacking_work_rate"].value_counts(dropna=False).rename_axis("attacking_work_rate").reset_index(name="rows"))
display(player_attributes["defensive_work_rate"].value_counts(dropna=False).rename_axis("defensive_work_rate").reset_index(name="rows"))


,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21,62.0,66.0,right,medium,medium,49.0,...,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0


,count,mean,std,min,25%,50%,75%,max
overall_rating,183142.0,68.60,7.04,33.0,64.0,69.0,73.0,94.0
potential,183142.0,73.46,6.59,39.0,69.0,74.0,78.0,97.0
crossing,183142.0,55.09,17.24,1.0,45.0,59.0,68.0,95.0
finishing,183142.0,49.92,19.04,1.0,34.0,53.0,65.0,97.0
heading_accuracy,183142.0,57.27,16.49,1.0,49.0,60.0,68.0,98.0
short_passing,183142.0,62.43,14.19,3.0,57.0,65.0,72.0,97.0
volleys,181265.0,49.47,18.26,1.0,35.0,52.0,64.0,93.0
dribbling,183142.0,59.18,17.74,1.0,52.0,64.0,72.0,97.0
curve,181265.0,52.97,18.26,2.0,41.0,56.0,67.0,94.0
free_kick_accuracy,183142.0,49.38,17.83,1.0,36.0,50.0,63.0,97.0


,preferred_foot,rows
0,right,138409
1,left,44733
2,None,836


,attacking_work_rate,rows
0,medium,125070
1,high,42823
2,low,8569
3,None,3639
4,None,3230
5,norm,348
6,y,106
7,le,104
8,stoc,89


,defensive_work_rate,rows
0,medium,130846
1,high,27041
2,low,18432
3,_0,2394
4,o,1550
5,None,836
6,1,441
7,ormal,348
8,2,342
9,3,258


In [23]:
# Premier League players appearing in match lineups
lineup_player_columns = [f"home_player_{i}" for i in range(1, 12)] + [f"away_player_{i}" for i in range(1, 12)]

lineup_union = "\nUNION ALL\n".join(
    f'''
    SELECT
        M.season,
        M.date,
        M.match_api_id,
        M.home_team_api_id,
        M.away_team_api_id,
        '{column}' AS lineup_slot,
        {column} AS player_api_id
    FROM Match M
    JOIN League L ON L.id = M.league_id
    WHERE L.name = 'England Premier League'
      AND {column} IS NOT NULL
    '''
    for column in lineup_player_columns
)

epl_lineup_players = pd.read_sql(
    f'''
    SELECT
        P.player_api_id,
        P.player_name,
        P.height,
        P.weight,
        COUNT(*) AS lineup_appearances,
        MIN(season) AS first_season,
        MAX(season) AS last_season
    FROM ({lineup_union}) lineups
    JOIN Player P ON P.player_api_id = lineups.player_api_id
    GROUP BY P.player_api_id, P.player_name
    ORDER BY lineup_appearances DESC
    ''',
    conn,
)

print(f"Distinct Premier League lineup players: {epl_lineup_players['player_api_id'].nunique():,}")
display(epl_lineup_players.head(20))


Distinct Premier League lineup players: 1,397


,player_api_id,player_name,height,weight,lineup_appearances,first_season,last_season
0,31465,Tim Howard,190.50,194,282,2008/2009,2015/2016
1,31432,Joe Hart,195.58,201,275,2008/2009,2015/2016
2,23782,Gareth Barry,182.88,174,258,2008/2009,2015/2016
3,40695,Ryan Shawcross,190.50,168,253,2008/2009,2015/2016
4,24846,Leighton Baines,170.18,154,251,2008/2009,2015/2016
5,30859,Petr Cech,195.58,198,251,2008/2009,2015/2016
6,30627,John Terry,187.96,198,243,2008/2009,2015/2016
7,31306,Branislav Ivanovic,185.42,201,234,2008/2009,2015/2016
8,23783,Gary Cahill,193.04,190,233,2008/2009,2015/2016
9,23268,Phil Jagielka,182.88,183,224,2008/2009,2015/2016


In [24]:
# Join each Premier League lineup player to their latest available attribute snapshot
latest_player_attributes = (
    player_attributes.sort_values("date")
    .groupby("player_api_id", as_index=False)
    .tail(1)
)

epl_players_latest_attributes = epl_lineup_players.merge(
    latest_player_attributes,
    on="player_api_id",
    how="left",
)

display(
    epl_players_latest_attributes[
        [
            "player_name",
            "lineup_appearances",
            "first_season",
            "last_season",
            "date",
            "overall_rating",
            "potential",
            "preferred_foot",
            "attacking_work_rate",
            "defensive_work_rate",
        ]
    ].head(20)
)

display(
    epl_players_latest_attributes[
        ["overall_rating", "potential", "height", "weight"]
    ].describe().round(2)
)


,player_name,lineup_appearances,first_season,last_season,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate
0,Tim Howard,282,2008/2009,2015/2016,2016-04-14,79.0,79.0,right,medium,medium
1,Joe Hart,275,2008/2009,2015/2016,2016-01-28,84.0,84.0,right,medium,medium
2,Gareth Barry,258,2008/2009,2015/2016,2015-09-21,79.0,79.0,left,medium,high
3,Ryan Shawcross,253,2008/2009,2015/2016,2015-09-21,80.0,80.0,right,medium,high
4,Leighton Baines,251,2008/2009,2015/2016,2016-04-07,82.0,82.0,left,high,medium
5,Petr Cech,251,2008/2009,2015/2016,2016-01-28,86.0,86.0,left,medium,medium
6,John Terry,243,2008/2009,2015/2016,2016-03-10,84.0,84.0,right,medium,high
7,Branislav Ivanovic,234,2008/2009,2015/2016,2016-04-21,80.0,80.0,right,high,medium
8,Gary Cahill,233,2008/2009,2015/2016,2016-04-21,82.0,82.0,right,medium,medium
9,Phil Jagielka,224,2008/2009,2015/2016,2015-09-21,81.0,81.0,right,medium,high


,overall_rating,potential,height,weight
count,1396.00,1396.00,1397.00,1397.00
mean,72.11,73.92,182.48,170.75
std,5.92,6.31,6.52,15.97
min,51.00,57.00,165.10,132.00
25%,68.00,69.00,177.80,159.00
50%,72.00,74.00,182.88,170.00
75%,76.00,78.00,187.96,181.00
max,93.00,93.00,203.20,225.00


In [25]:
# Save Premier League-only player data to CSV
# One row per player attribute snapshot for players who appear in Premier League lineups.
epl_player_data = (
    epl_lineup_players[
        [
            "player_api_id",
            "player_name",
            "height",
            "weight",
            "lineup_appearances",
            "first_season",
            "last_season",
        ]
    ]
    .merge(
        players[["player_api_id", "player_fifa_api_id", "birthday"]],
        on="player_api_id",
        how="left",
    )
    .merge(
        player_attributes,
        on=["player_api_id", "player_fifa_api_id"],
        how="left",
        suffixes=("", "_attribute"),
    )
)

epl_player_data_path = DATA_DIR / "epl_player_data.csv"
epl_player_data.to_csv(epl_player_data_path, index=False)

print(
    f"Saved {len(epl_player_data):,} attribute rows for "
    f"{epl_player_data['player_api_id'].nunique():,} Premier League players to {epl_player_data_path}"
)
display(epl_player_data.head())


Saved 32,708 attribute rows for 1,397 Premier League players to c:\Users\Madsl\Dropbox\DTU\Kand\3.Semester\MBML\Project_MBML\data\epl_player_data.csv


,player_api_id,player_name,height,weight,lineup_appearances,first_season,last_season,player_fifa_api_id,birthday,id,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,31465,Tim Howard,190.5,194,282,2008/2009,2015/2016,16254,1979-03-06,171738,...,52.0,43.0,11.0,12.0,14.0,82.0,76.0,77.0,78.0,79.0
1,31465,Tim Howard,190.5,194,282,2008/2009,2015/2016,16254,1979-03-06,171739,...,52.0,43.0,11.0,12.0,14.0,82.0,76.0,77.0,81.0,82.0
2,31465,Tim Howard,190.5,194,282,2008/2009,2015/2016,16254,1979-03-06,171740,...,25.0,42.0,25.0,25.0,25.0,81.0,75.0,76.0,80.0,81.0
3,31465,Tim Howard,190.5,194,282,2008/2009,2015/2016,16254,1979-03-06,171741,...,25.0,42.0,25.0,25.0,25.0,82.0,76.0,76.0,80.0,81.0
4,31465,Tim Howard,190.5,194,282,2008/2009,2015/2016,16254,1979-03-06,171742,...,25.0,42.0,25.0,25.0,25.0,83.0,77.0,76.0,82.0,82.0


## Betting odds data

The betting odds are stored directly in the `Match` table. Each bookmaker has three columns:

- `H`: home win odds
- `D`: draw odds
- `A`: away win odds

For Premier League matches, the database contains 3,040 matches. The most complete bookmaker columns are `B365`, `WH`, and `VC`, each available for all 3,040 matches.


In [26]:
# Identify betting odds columns in the Match table
match_schema = pd.read_sql("PRAGMA table_info(Match);", conn)

odds_columns = [
    col for col in match_schema["name"]
    if col.endswith(("H", "D", "A")) and col not in ["id"]
]

bookmakers = sorted({col[:-1] for col in odds_columns})

print(f"Number of odds columns: {len(odds_columns)}")
print(f"Bookmakers: {bookmakers}")
display(match_schema[match_schema["name"].isin(odds_columns)])


Number of odds columns: 30
Bookmakers: ['B365', 'BS', 'BW', 'GB', 'IW', 'LB', 'PS', 'SJ', 'VC', 'WH']


,cid,name,type,notnull,dflt_value,pk
85,85,B365H,NUMERIC,0,None,0
86,86,B365D,NUMERIC,0,None,0
87,87,B365A,NUMERIC,0,None,0
88,88,BWH,NUMERIC,0,None,0
89,89,BWD,NUMERIC,0,None,0
90,90,BWA,NUMERIC,0,None,0
91,91,IWH,NUMERIC,0,None,0
92,92,IWD,NUMERIC,0,None,0
93,93,IWA,NUMERIC,0,None,0
94,94,LBH,NUMERIC,0,None,0


In [27]:
# Coverage of betting columns for Premier League matches
coverage_sql = '''
SELECT
    COUNT(*) AS epl_matches,
    {coverage_columns}
FROM Match M
JOIN League L ON L.id = M.league_id
WHERE L.name = 'England Premier League'
'''.format(
    coverage_columns=",\n    ".join(
        f"SUM(CASE WHEN {col} IS NOT NULL THEN 1 ELSE 0 END) AS {col}_rows"
        for col in odds_columns
    )
)

betting_coverage = pd.read_sql(coverage_sql, conn)
display(betting_coverage.T.rename(columns={0: "non_null_rows"}))

bookmaker_coverage = []
for bookmaker in bookmakers:
    home_col = f"{bookmaker}H"
    draw_col = f"{bookmaker}D"
    away_col = f"{bookmaker}A"
    complete_triplets = pd.read_sql(
        f'''
        SELECT COUNT(*) AS complete_triplets
        FROM Match M
        JOIN League L ON L.id = M.league_id
        WHERE L.name = 'England Premier League'
          AND {home_col} IS NOT NULL
          AND {draw_col} IS NOT NULL
          AND {away_col} IS NOT NULL
        ''',
        conn,
    ).iloc[0, 0]
    bookmaker_coverage.append(
        {
            "bookmaker": bookmaker,
            "complete_triplets": complete_triplets,
            "coverage_pct": round(100 * complete_triplets / betting_coverage.loc[0, "epl_matches"], 2),
        }
    )

bookmaker_coverage = pd.DataFrame(bookmaker_coverage).sort_values(
    ["complete_triplets", "bookmaker"],
    ascending=[False, True],
)
display(bookmaker_coverage)


,non_null_rows
epl_matches,3040
B365H_rows,3040
B365D_rows,3040
B365A_rows,3040
BWH_rows,3039
BWD_rows,3039
BWA_rows,3039
IWH_rows,3038
IWD_rows,3038
IWA_rows,3038


,bookmaker,complete_triplets,coverage_pct
0,B365,3040,100.00
8,VC,3040,100.00
9,WH,3040,100.00
2,BW,3039,99.97
5,LB,3039,99.97
4,IW,3038,99.93
7,SJ,2320,76.32
1,BS,1900,62.50
3,GB,1899,62.47
6,PS,1519,49.97


In [28]:
# Load clean Premier League match-level betting data
odds_select = ",\n    ".join(f"M.{col}" for col in odds_columns)

epl_betting_query = f'''
SELECT
    M.id,
    M.match_api_id,
    M.season,
    M.stage,
    M.date,
    HT.team_long_name AS home_team,
    AT.team_long_name AS away_team,
    M.home_team_goal,
    M.away_team_goal,
    CASE
        WHEN M.home_team_goal > M.away_team_goal THEN 'H'
        WHEN M.home_team_goal = M.away_team_goal THEN 'D'
        ELSE 'A'
    END AS result,
    {odds_select}
FROM Match M
JOIN League L ON L.id = M.league_id
JOIN Team HT ON HT.team_api_id = M.home_team_api_id
JOIN Team AT ON AT.team_api_id = M.away_team_api_id
WHERE L.name = 'England Premier League'
ORDER BY M.date
'''

epl_betting_data = pd.read_sql(epl_betting_query, conn, parse_dates=["date"])
display(epl_betting_data.head())
display(epl_betting_data[["B365H", "B365D", "B365A", "WHH", "WHD", "WHA", "VCH", "VCD", "VCA"]].describe().round(2))


,id,match_api_id,season,stage,date,home_team,away_team,home_team_goal,away_team_goal,result,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1730,489043,2008/2009,1,2008-08-16,Arsenal,West Bromwich Albion,1,0,H,...,12.00,1.25,6.0,13.00,1.22,5.50,13.00,1.22,5.00,13.00
1,1731,489044,2008/2009,1,2008-08-16,Sunderland,Liverpool,0,1,A,...,1.73,5.50,3.8,1.65,5.00,3.40,1.70,4.50,3.40,1.73
2,1732,489045,2008/2009,1,2008-08-16,West Ham United,Wigan Athletic,2,1,H,...,3.75,1.90,3.5,4.35,1.91,3.25,4.00,1.91,3.25,3.80
3,1734,489047,2008/2009,1,2008-08-16,Everton,Blackburn Rovers,2,3,A,...,3.40,2.05,3.3,4.00,2.00,3.25,3.75,2.00,3.25,3.50
4,1735,489048,2008/2009,1,2008-08-16,Middlesbrough,Tottenham Hotspur,2,1,H,...,2.20,3.20,3.4,2.30,3.00,3.25,2.30,2.80,3.25,2.30


,B365H,B365D,B365A,WHH,WHD,WHA,VCH,VCD,VCA
count,3040.00,3040.00,3040.00,3040.00,3040.00,3040.00,3040.00,3040.00,3040.00
mean,2.70,3.95,4.91,2.66,3.67,4.70,2.71,3.96,4.96
std,1.69,1.00,3.91,1.59,0.81,3.63,1.69,1.01,3.98
min,1.10,3.00,1.22,1.10,2.80,1.22,1.09,2.50,1.22
25%,1.67,3.30,2.50,1.67,3.20,2.50,1.67,3.30,2.50
50%,2.20,3.60,3.50,2.15,3.30,3.40,2.20,3.60,3.60
75%,3.00,4.20,5.50,2.90,3.80,5.50,3.00,4.10,5.50
max,15.00,11.00,29.00,12.00,9.50,26.00,15.00,10.00,29.00


In [29]:
# Add implied probabilities for the most complete bookmaker odds
for bookmaker in ["B365", "WH", "VC"]:
    home_col = f"{bookmaker}H"
    draw_col = f"{bookmaker}D"
    away_col = f"{bookmaker}A"

    epl_betting_data[f"{bookmaker}_home_implied_prob"] = 1 / epl_betting_data[home_col]
    epl_betting_data[f"{bookmaker}_draw_implied_prob"] = 1 / epl_betting_data[draw_col]
    epl_betting_data[f"{bookmaker}_away_implied_prob"] = 1 / epl_betting_data[away_col]
    epl_betting_data[f"{bookmaker}_bookmaker_margin"] = (
        epl_betting_data[f"{bookmaker}_home_implied_prob"]
        + epl_betting_data[f"{bookmaker}_draw_implied_prob"]
        + epl_betting_data[f"{bookmaker}_away_implied_prob"]
        - 1
    )

margin_columns = [f"{bookmaker}_bookmaker_margin" for bookmaker in ["B365", "WH", "VC"]]
display(epl_betting_data[margin_columns].describe().round(4))


,B365_bookmaker_margin,WH_bookmaker_margin,VC_bookmaker_margin
count,3040.0000,3040.0000,3040.0000
mean,0.0423,0.0658,0.0386
std,0.0150,0.0129,0.0193
min,0.0170,0.0292,0.0021
25%,0.0258,0.0604,0.0253
50%,0.0505,0.0633,0.0308
75%,0.0556,0.0667,0.0463
max,0.0977,0.1930,0.1429


In [30]:
# Save Premier League-only betting data to CSV
epl_betting_data_path = DATA_DIR / "epl_betting_data.csv"
epl_betting_data.to_csv(epl_betting_data_path, index=False)

print(f"Saved {len(epl_betting_data):,} Premier League matches to {epl_betting_data_path}")
display(epl_betting_data.head())


Saved 3,040 Premier League matches to c:\Users\Madsl\Dropbox\DTU\Kand\3.Semester\MBML\Project_MBML\data\epl_betting_data.csv


,id,match_api_id,season,stage,date,home_team,away_team,home_team_goal,away_team_goal,result,...,B365_away_implied_prob,B365_bookmaker_margin,WH_home_implied_prob,WH_draw_implied_prob,WH_away_implied_prob,WH_bookmaker_margin,VC_home_implied_prob,VC_draw_implied_prob,VC_away_implied_prob,VC_bookmaker_margin
0,1730,489043,2008/2009,1,2008-08-16,Arsenal,West Bromwich Albion,1,0,H,...,0.066667,0.053846,0.854701,0.181818,0.083333,0.119852,0.800000,0.166667,0.076923,0.043590
1,1731,489044,2008/2009,1,2008-08-16,Sunderland,Liverpool,0,1,A,...,0.598802,0.058398,0.181818,0.303030,0.636943,0.121791,0.181818,0.263158,0.606061,0.051037
2,1732,489045,2008/2009,1,2008-08-16,West Ham United,Wigan Athletic,2,1,H,...,0.238095,0.055773,0.546448,0.312500,0.266667,0.125615,0.526316,0.285714,0.229885,0.041915
3,1734,489047,2008/2009,1,2008-08-16,Everton,Blackburn Rovers,2,3,A,...,0.250000,0.053030,0.512821,0.322581,0.285714,0.121115,0.487805,0.303030,0.250000,0.040835
4,1735,489048,2008/2009,1,2008-08-16,Middlesbrough,Tottenham Hotspur,2,1,H,...,0.444444,0.051062,0.344828,0.312500,0.465116,0.122444,0.312500,0.294118,0.434783,0.041400
